# Manual testing: newly added visualization capabilities

This notebook exercises the recently added plot capabilities against **real data**, loaded exactly
the way the application loads it (via `_get_variable_data`).

Each section follows the same recipe: **load real data → build a minimal `Algorithm` → execute it →
display the result.**

Capabilities covered:

1. **Wind vector plots** (`WindVectorPlotSpec`) — arrows / barbs colored by speed, plus an animation over lead time.
2. **Ensemble plume plots** (`EnsemblePlumePlotSpec`) — member spaghetti + mean + spread band.
3. **Contour maps** (`ContourMapPlotSpec`) — filled / line iso-contours.
4. **Bonus:** compute + visualize composition (`ComputeLinearTrend` → contour map).


## Prerequisites

- Select the project **`.venv`** as the notebook kernel.
- Launch Jupyter from the repo root so `import llm_agent` and `import tests` resolve.
- Real data lives in **S3** — make sure AWS credentials are available in the environment.
- A **local Dask cluster** is started below to compute chunked arrays.


## Setup


In [ ]:
# ruff: noqa: T201

import typing
from pathlib import Path
from unittest.mock import Mock

import xarray as xr
from dask.distributed import Client, LocalCluster
from IPython.display import Image, Video

# Importing the exec package registers ALL execution handlers via singledispatch
# (wind, ensemble, contour, sst, data-array ops, animations, ...). One import is enough.
import llm_agent.agents.algorithm_agent.algorithm.exec  # pyright: ignore[reportUnusedImport] # noqa: F401
from llm_agent.agents.algorithm_agent.algorithm.base import Algorithm, make_function_call_model
from llm_agent.agents.algorithm_agent.algorithm.exec.base import ExecutionConfig, execute_algorithm
from llm_agent.agents.algorithm_agent.algorithm.function_sets.base import FunctionSet
from llm_agent.agents.algorithm_agent.algorithm.function_sets.data_array_functions import (
    SpatioTemporalDataArrayFunctionSet,
)
from llm_agent.agents.algorithm_agent.algorithm.function_sets.plot.contour import (
    ContourPlottingFunctionSet,
)
from llm_agent.agents.algorithm_agent.algorithm.function_sets.plot.ensemble import (
    EnsemblePlottingFunctionSet,
)
from llm_agent.agents.algorithm_agent.algorithm.function_sets.plot.wind import (
    WindPlottingFunctionSet,
)
from llm_agent.context import ChatContext, DataRef, LLMAgentContext, ToolCallContext
from llm_agent.dao.knowledge_graph_dao import KnowledgeGraphDAO
from llm_agent.tools.data_analysis_tool.data_analysis_tool import (
    _get_variable_data,  # pyright: ignore[reportPrivateUsage]
)
from llm_agent.tools.data_analysis_tool.types import AnimationRef, ImageRef


In [ ]:
# client = Client("tcp://127.0.0.1:8786", asynchronous=True)
cluster = LocalCluster()
client = Client(cluster.scheduler_address, asynchronous=True)
client

In [ ]:
class NotebookChatContext(ChatContext):
    """Minimal in-memory ChatContext: stores persisted bytes and hands them back.

    The exec handlers write images/animations to a temp file and pass the path to persist_data;
    we read the bytes immediately (before the temp file is cleaned up) so results stay displayable.
    """

    def __init__(self, global_context: LLMAgentContext) -> None:
        self._global_context = global_context
        self.data_store: dict[str, object] = {}

    @property
    def global_context(self) -> LLMAgentContext:
        return self._global_context

    def persist_data(self, data_or_path: object, *, data_type: str) -> str:
        key = f"{data_type}_{len(self.data_store)}"
        if isinstance(data_or_path, str) and data_type in ("image", "animation"):
            with open(data_or_path, "rb") as f:
                self.data_store[key] = f.read()
        else:
            self.data_store[key] = data_or_path
        return key

    def retrieve_data(self, data_ref: DataRef) -> typing.Any:  # noqa: ANN401
        return self.data_store[data_ref.key]


dao = KnowledgeGraphDAO()  # bundled .ttl knowledge graphs
global_context = LLMAgentContext(
    place_lookup=Mock(),  # only needed for geocoding functions (unused here)
    knowledge_graph_dao=dao,
    dask_client=client,
    bedrock_client=Mock(),  # only needed for LLM calls (unused here)
)
chat_context = NotebookChatContext(global_context)
tool_call_context = ToolCallContext(tool_call_id="notebook", chat_context=chat_context)

In [ ]:
def function_names(function_set: FunctionSet) -> list[str]:
    """The exact func_name strings an algorithm may use with this function set."""
    return [
        typing.get_args(f.model_fields["func_name"].annotation)[0] for f in function_set.functions
    ]


async def run_algorithm(
    function_set: FunctionSet,
    statements: list[dict[str, typing.Any]],
    output_names: list[str],
    input_vars: dict[str, typing.Any],
    exec_config: ExecutionConfig | None = None,
):
    """Build an Algorithm from raw statement dicts and execute it."""
    function_call_model = make_function_call_model(function_set.functions)
    algo = Algorithm[function_call_model].model_validate(
        {
            "statements": statements,
            "output_variables": [{"type": "VariableReference", "name": n} for n in output_names],
        }
    )
    return await execute_algorithm(
        tool_call_context, algo, input_vars, exec_config or ExecutionConfig()
    )


def show(result: dict[str, typing.Any], name: str) -> Video | None:
    """Display an ImageRef/AnimationRef result inline (or print scalars)."""
    ref = result[name]
    match ref:
        case ImageRef() | AnimationRef(format="gif"):
            display(Image(data=chat_context.retrieve_data(ref)))
        case AnimationRef(format="mp4", name=name):
            path = Path("./data") / name
            path.parent.mkdir(parents=True, exist_ok=True)
            with open(path, "wb") as f:
                f.write(chat_context.retrieve_data(ref))
            return Video(path, html_attributes="controls muted autoplay loop")
        case _:
            print(name, "=", ref)
    return None


def reduce_to_latlon(da: xr.DataArray, **sel: typing.Any) -> xr.DataArray:  # noqa: ANN401
    """Label-select, then collapse every remaining non-lat/lon dim to its first index."""
    if sel:
        da = da.sel(**sel)  # pyright: ignore[reportUnknownMemberType]
    return da.isel({d: 0 for d in da.dims if d not in ("lat", "lon")})

## Discover available variables

List the datasets in the knowledge graph and the variable IDs / standard names / dimensions.
Use this to pick real variable IDs (and to sanity-check dimension names such as `ensemble_member`).


In [ ]:
def find_variables(
    field_name: str | None = None,
    require_dims: tuple[str, ...] = (),
    dataset_substr: str | None = None,
) -> list[tuple[str, str, list[str]]]:
    """Find (variable_id, dataset_id, dim_field_names) tuples matching the criteria."""
    matches: list[tuple[str, str, list[str]]] = []
    for ds in dao.fetch_datasets():
        if dataset_substr and dataset_substr.lower() not in ds.id.lower():
            continue
        for v in ds.variables:
            dims = [d.field_name for d in v.dimensions]
            if field_name and v.field_name != field_name:
                continue
            if not all(rd in dims for rd in require_dims):
                continue
            matches.append((v.id, ds.id, dims))
    return matches


for ds in dao.fetch_datasets():
    print(ds.id)
    for v in ds.variables:
        dims = ",".join(d.field_name for d in v.dimensions)
        print(f"    {v.id}  [field={v.field_name}, std={v.standard_name}, dims=({dims})]")


## 1. Wind vector plot

Load 10 m eastward/northward wind components from an ensemble forecast, reduce to a single 2D
(lat, lon) field over a region, and render a wind vector map colored by speed.


In [ ]:
# Pick a u/v pair from the same ensemble-forecast dataset.
u_matches = find_variables(field_name="wind_u_10m", require_dims=("ensemble_member",))
v_matches = find_variables(field_name="wind_v_10m", require_dims=("ensemble_member",))
u_id, wind_ds_id, u_dims = u_matches[0]
v_id = next(vid for vid, ds_id, _ in v_matches if ds_id == wind_ds_id)
print("dataset:", wind_ds_id)
print("u:", u_id, "\nv:", v_id, "\ndims:", u_dims)

u_full = await _get_variable_data(dao, u_id)
v_full = await _get_variable_data(dao, v_id)
u_full

In [ ]:
# North Atlantic region, single init/lead/member -> 2D (lat, lon), then load into memory.
region = {"lat": slice(20, 30), "lon": slice(-30, -10)}
u2d = reduce_to_latlon(u_full, **region)
v2d = reduce_to_latlon(v_full, **region)
print(u2d.sizes)

In [ ]:
wind_stmt = {
    "type": "VariableAssignment",
    "name": "wind_plot",
    "value": {
        "type": "FunctionCall",
        "func": {
            "type": "Function",
            "func_name": "MakePlot",
            "plot_spec": {
                "spec_name": "WindVectorPlotSpec",
                "u_data_array": {"type": "VariableReference", "name": "u"},
                "v_data_array": {"type": "VariableReference", "name": "v"},
                "color_by_speed": True,
                "subsample": 3,
                "title": "10m Wind",
            },
        },
    },
}
result = await run_algorithm(
    WindPlottingFunctionSet(), [wind_stmt], ["wind_plot"], {"u": u2d, "v": v2d}
)
show(result, "wind_plot")

### Wind animation over lead time

Keep the `lead_time` dimension (first several steps) and animate the evolving wind field.


In [ ]:
def reduce_keep(da: xr.DataArray, keep: str, **sel: typing.Any) -> xr.DataArray:  # noqa: ANN401
    if sel:
        da = da.sel(**sel)  # pyright: ignore[reportUnknownMemberType]
    return da.isel({d: 0 for d in da.dims if d not in ("lat", "lon", keep)})


u_anim = reduce_keep(u_full, "lead_time", **region).isel(lead_time=slice(0, 8))
v_anim = reduce_keep(v_full, "lead_time", **region).isel(lead_time=slice(0, 8))
print(u_anim.sizes)

wind_anim_stmt = {
    "type": "VariableAssignment",
    "name": "wind_anim",
    "value": {
        "type": "FunctionCall",
        "func": {
            "type": "Function",
            "func_name": "MakeAnimation",
            "animation_spec": {
                "plot_spec": {
                    "spec_name": "WindVectorPlotSpec",
                    "u_data_array": {"type": "VariableReference", "name": "u"},
                    "v_data_array": {"type": "VariableReference", "name": "v"},
                    "color_by_speed": True,
                    "subsample": 3,
                    "title": "10m Wind",
                },
                "frame_dim": "lead_time",
                "frame_interval_seconds": 0.4,
            },
        },
    },
}
result = await run_algorithm(
    WindPlottingFunctionSet(), [wind_anim_stmt], ["wind_anim"], {"u": u_anim, "v": v_anim}
)
show(result, "wind_anim")

In [ ]:
show(result, "wind_anim")

## 2. Ensemble plume plot

Select an ensemble forecast variable at a single point so the data is 2D
`(ensemble_member, lead_time)`, then draw the plume: member lines, ensemble mean, and a spread band.


In [ ]:
temp_matches = find_variables(field_name="temperature_2m", require_dims=("ensemble_member",))
temp_id, temp_ds_id, temp_dims = temp_matches[0]
print("dataset:", temp_ds_id, "\nvar:", temp_id, "\ndims:", temp_dims)

temp_full = await _get_variable_data(dao, temp_id)

# Nearest grid point to ~ New York Harbor, single init time -> (ensemble_member, lead_time).
point = temp_full.sel(lat=40.7, lon=-74.0, method="nearest")  # pyright: ignore[reportUnknownMemberType]
plume_da = point.isel({d: 0 for d in point.dims if d not in ("ensemble_member", "lead_time")})
print(plume_da.sizes)

plume_stmt = {
    "type": "VariableAssignment",
    "name": "plume",
    "value": {
        "type": "FunctionCall",
        "func": {
            "type": "Function",
            "func_name": "MakePlot",
            "plot_spec": {
                "spec_name": "EnsemblePlumePlotSpec",
                "data_array": {"type": "VariableReference", "name": "da"},
                "band_type": "p10_p90",
                "annotation": {"title": "2m temperature ensemble @ (40.7N, 74W)", "ylabel": "K"},
            },
        },
    },
}
result = await run_algorithm(
    EnsemblePlottingFunctionSet(), [plume_stmt], ["plume"], {"da": plume_da}
)
show(result, "plume")

## 3. Contour map

Load a sea surface temperature field for a single day and region and render it as filled contours.


In [ ]:
sst_matches = find_variables(field_name="analysed_sst")
sst_id, sst_ds_id, sst_dims = sst_matches[0]
print("dataset:", sst_ds_id, "\nvar:", sst_id, "\ndims:", sst_dims)

sst_full = await _get_variable_data(dao, sst_id)

In [ ]:
# Single day, North Atlantic region -> 2D (lat, lon).
sst2d = sst_full.sel(time="2023-08-01", method="nearest").sel(**region)  # pyright: ignore[reportArgumentType, reportUnknownMemberType]
print(sst2d.sizes)

contour_stmt = {
    "type": "VariableAssignment",
    "name": "contour",
    "value": {
        "type": "FunctionCall",
        "func": {
            "type": "Function",
            "func_name": "MakePlot",
            "plot_spec": {
                "spec_name": "ContourMapPlotSpec",
                "data_array": {"type": "VariableReference", "name": "da"},
                "filled": True,
                "levels": 12,
                "title": "SST contours (2023-08-01)",
            },
        },
    },
}
result = await run_algorithm(
    ContourPlottingFunctionSet(), [contour_stmt], ["contour"], {"da": sst2d}
)
show(result, "contour")

## 4. Bonus: compute + visualize composition

A two-statement algorithm that first computes a per-pixel linear **trend** over time
(`ComputeLinearTrend`) and then draws the trend map as an unfilled contour plot. This exercises a
compute capability and a plot capability composed in a single algorithm.

The SST time series is passed **lazily** (chunked Dask array) so the local cluster actually does the
`polyfit`; only the small 2D trend map is materialized for plotting.


In [ ]:
# Small region + one year of daily SST, kept lazy (Dask-backed).
trend_region = {"lat": slice(25, 45), "lon": slice(-80, -50)}
sst_series = sst_full.sel(time=slice("2023-01-01", "2023-12-31")).sel(**trend_region)  # pyright: ignore[reportArgumentType, reportUnknownMemberType]
print("lazy input sizes:", sst_series.sizes)

# Merge the spatiotemporal data functions (for ComputeLinearTrend) with contour plotting.
trend_fs = FunctionSet.merge([SpatioTemporalDataArrayFunctionSet(), ContourPlottingFunctionSet()])
print("ComputeLinearTrend available:", "ComputeLinearTrend" in function_names(trend_fs))

trend_statements = [
    {
        "type": "VariableAssignment",
        "name": "trend",
        "value": {
            "type": "FunctionCall",
            "func": {
                "type": "Function",
                "func_name": "ComputeLinearTrend",
                "data_array": {"type": "VariableReference", "name": "sst"},
                "time_unit": "decade",
                "time_dim": "time",
            },
        },
    },
    {
        "type": "VariableAssignment",
        "name": "trend_plot",
        "value": {
            "type": "FunctionCall",
            "func": {
                "type": "Function",
                "func_name": "MakePlot",
                "plot_spec": {
                    "spec_name": "ContourMapPlotSpec",
                    "data_array": {"type": "VariableReference", "name": "trend"},
                    "filled": True,
                    "levels": 12,
                    "cmap": "RdBu_r",
                    "title": "2023 SST trend (K / decade)",
                },
            },
        },
    },
]
result = await run_algorithm(trend_fs, trend_statements, ["trend_plot"], {"sst": sst_series})
show(result, "trend_plot")

## Cleanup


In [ ]:
await client.close()  # pyright: ignore[reportGeneralTypeIssues, reportUnknownMemberType]